# AIND Ephys Preprocessing

Run [aind-ephys-preprocessing](https://github.com/AllenNeuralDynamics/aind-ephys-preprocessing/blob/main/code/run_capsule.py) on the dispatched job from `01_aind_ephys_dispatch.ipynb`, against a 1-second window of the toy recording.

The preprocessing capsule reads `job_*.json` files from `../data/` (relative to its `code/` directory) and writes preprocessed recordings + metadata to `../results/`. The default `params.json` requires recordings of at least 120 s and runs motion correction; for the toy data (10 s, 16 channels) we override those via a custom params JSON and clip to 1 s with `--t-start 0 --t-stop 1`.

## Imports and prerequisites

In [1]:
import json
import shutil
import subprocess
from pathlib import Path

## Install extra preprocessing dependencies

The capsule depends on `aind-data-schema` (for the processing-metadata records it writes) and `scikit-learn` (used by bad-channel detection). The toy recording is uncompressed, so we don't need `wavpack-numcodecs`.

In [2]:
import sys
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "aind-data-schema", "scikit-learn",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 19 packages in 15ms
Uninstalled 2 packages in 7ms
Installed 2 packages in 2ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'aind-data-schema', 'scikit-learn'], returncode=0)

## Clone the preprocessing capsule and seed `data/` with the dispatch job

The dispatcher's `job_0.json` (already produced by `01_aind_ephys_dispatch.ipynb` and saved next to this notebook in `dispatch_results/`) is what the preprocessor consumes.

In [3]:
preprocessing_repo = Path("/tmp/aind-ephys-preprocessing")
if not preprocessing_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-preprocessing.git",
            str(preprocessing_repo),
        ],
        check=True,
    )

data_dir = preprocessing_repo / "data"
results_dir = preprocessing_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.glob("job_*.json")) + list(results_dir.iterdir()):
    if stale.is_dir():
        shutil.rmtree(stale)
    else:
        stale.unlink()

dispatch_job = Path.cwd() / "dispatch_results" / "job_0.json"
assert dispatch_job.exists(), (
    f"{dispatch_job} not found. Run 01_aind_ephys_dispatch.ipynb first."
)
shutil.copy2(dispatch_job, data_dir / "job_0.json")
print("Seeded:", data_dir / "job_0.json")

Cloning into '/tmp/aind-ephys-preprocessing'...


Seeded: /tmp/aind-ephys-preprocessing/data/job_0.json


## Write a small-window params.json

Starts from the upstream defaults but lowers `min_preprocessing_duration` to 0.5 s and skips motion correction (dredge needs longer than 1 s of data).

In [4]:
upstream_params = json.loads(
    (preprocessing_repo / "code" / "params.json").read_text()
)
params = {**upstream_params, "min_preprocessing_duration": 0.5}
params["job_kwargs"] = {**params.get("job_kwargs", {}), "n_jobs": 1}
# When --params is passed, motion settings come from the JSON (`compute`/`apply`
# are popped out of motion_correction). Disable both for the toy run.
params["motion_correction"] = {
    **params.get("motion_correction", {}),
    "compute": False,
    "apply": False,
}

params_path = preprocessing_repo / "code" / "params_toy.json"
params_path.write_text(json.dumps(params, indent=2))
print("Wrote", params_path)

Wrote /tmp/aind-ephys-preprocessing/code/params_toy.json


## Run the preprocessing capsule

We invoke `run_capsule.py` from the capsule's `code/` directory so its `../data` and `../results` paths resolve. We pass `--t-start 0 --t-stop 1` to clip to a 1-second test window and `--motion skip` to bypass motion correction. Note: `--params` overrides every other flag, so we pass it last alongside `--motion skip` set inside the JSON.

In [5]:
argv = [
    "python", "-u", "run_capsule.py",
    "--params", str(params_path.name),
    "--t-start", "0",
    "--t-stop", "1",
    "--n-jobs", "1",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=preprocessing_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"preprocessing run failed with code {result.returncode}")

Running: python -u run_capsule.py --params params_toy.json --t-start 0 --t-stop 1 --n-jobs 1


Running preprocessing with the following parameters:
	DENOISING_STRATEGY: cmr
	FILTER TYPE: highpass
	REMOVE_OUT_CHANNELS: True
	REMOVE_BAD_CHANNELS: True
	MAX BAD CHANNEL FRACTION: 0.5
	COMPUTE_MOTION: False
	APPLY_MOTION: False
	MOTION PRESET: dredge_fast
	MOTION TEMPORAL BIN S: 1
	T_START: 0
	T_STOP: 1
	MIN_DURATION FOR PREPROCESSING: 0.5
	N_JOBS: 1
Found 1 configurations


PREPROCESSING
Preprocessing recording: session0 - block0_None_recording1
	Original recording duration: 10.0 -- Clipping to 0.0-1.0 s
	Duration: 1.0 s
	Bad channel detection:
		- dead channels - 0
		- noise channels - 0
		- out channels - 0
	Removing 0 out channels
	Highpass spatial filter failed: 'n_channel_pad' must be less than the number of channels in recording..
	Removing 0 channels after cmr preprocessing
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=30,000 - chunk_memory=1.83 MiB - total_memory=1.83 MiB - chunk_duration=1.00s
PREPROCESSING time: 1.2s



## Copy outputs next to the notebook

The capsule writes everything into `../results/`. We copy that tree into `./preprocessing_results/`.

In [6]:
local_results_dir = Path.cwd() / "preprocessing_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.rglob("*")):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  binary_block0_None_recording1.json  (2954 bytes)
  data_process_preprocessing_block0_None_recording1.json  (3055 bytes)
  preprocessed_block0_None_recording1  (dir)
  preprocessed_block0_None_recording1/binary.json  (2206 bytes)
  preprocessed_block0_None_recording1/probe.json  (8472 bytes)
  preprocessed_block0_None_recording1/properties  (dir)
  preprocessed_block0_None_recording1/properties/contact_vector.npy  (13824 bytes)
  preprocessed_block0_None_recording1/properties/gain_to_uV.npy  (256 bytes)
  preprocessed_block0_None_recording1/properties/group.npy  (256 bytes)
  preprocessed_block0_None_recording1/properties/location.npy  (384 bytes)
  preprocessed_block0_None_recording1/properties/offset_to_uV.npy  (256 bytes)
  preprocessed_block0_None_recording1/provenance.json  (93783 bytes)
  preprocessed_block0_None_recording1/si_folder.json  (2920 bytes)
  preprocessed_block0_None_recording1/traces_cached_seg0.raw  (1920064 bytes)
  preprocessed_block0_None_recording1.json  (93780

## Load the preprocessed recording and confirm it's 1 second long

In [7]:
import spikeinterface.full as si

preprocessed_dirs = [p for p in local_results_dir.iterdir() if p.is_dir() and "preprocessed" in p.name]
print("Preprocessed recording dirs:", [p.name for p in preprocessed_dirs])

if preprocessed_dirs:
    rec = si.load(preprocessed_dirs[0])
    print(rec)
    print("duration_s:", rec.get_total_duration())
    print("num_channels:", rec.get_num_channels())
    print("sampling_frequency:", rec.get_sampling_frequency())

Preprocessed recording dirs: ['preprocessed_block0_None_recording1']
BinaryFolderRecording: 16 channels - 30.0kHz - 1 segments - 30,001 samples - 1.00s - float32 dtype 
                       1.83 MiB
duration_s: 1.0000333333333333
num_channels: 16
sampling_frequency: 30000.0
